# Environment

In [1]:
! pip install pandas
! pip install pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 220.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 124.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
! pip install transformers accelerate bitsandbytes safetensors torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 15.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 250.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 109.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 31.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 234.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 189.3 MB/s eta 0:00:00a 0:00:01

In [3]:
! pip install huggingface-hub


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


## Import Library

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,pipeline
from tqdm import tqdm
import pandas as pd
import json

# Load Model

In [5]:
! nvidia-smi

Wed Jun  4 01:14:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.05             Driver Version: 550.127.05     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:98:00.0 Off |                    0 |
|  0%   29C    P8             21W /  300W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# token = "HF_TOKEN_REDACTED"

In [6]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [7]:
"""from huggingface_hub import login
import os

# Get HF token injected from RunPod secret
hf_token = os.getenv('HF_TOKEN')

# Login
login(token=hf_token)

print("Hugging Face login successful via RunPod Secret!")"""

'from huggingface_hub import login\nimport os\n\n# Get HF token injected from RunPod secret\nhf_token = os.getenv(\'HF_TOKEN\')\n\n# Login\nlogin(token=hf_token)\n\nprint("Hugging Face login successful via RunPod Secret!")'

## Full Model

In [8]:
from transformers import MllamaForConditionalGeneration, AutoProcessor
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

# Load model
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    low_cpu_mem_usage=True,
)

min_pixels = 224 * 28 * 28              
max_pixels = 768 * 28 * 28  

processor = AutoProcessor.from_pretrained(
    model_id, min_pixels=min_pixels, max_pixels=max_pixels
)

config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/89.4k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.47G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.09k [00:00<?, ?B/s]

### Testing Response with Images

In [12]:
# multiple images

In [ ]:
from PIL import Image
import requests
import torch
import gc

# Helper to resize images for VRAM optimization
def resize_images(image_list, size=(224, 224)):
    return [img.resize(size, Image.BILINEAR) for img in image_list]

# Load and resize images
image_urls = [
    "https://media.istockphoto.com/id/1018245126/photo/siberian-tiger-portrait.jpg?s=612x612&w=0&k=20&c=ONrnG-JrcVuqdtIjfr4Pj85ZswUPt8wD-VbJOnWYPw0=",
    "https://images.pexels.com/photos/458991/pexels-photo-458991.jpeg?cs=srgb&dl=pexels-pixabay-458991.jpg&fm=jpg"
]
original_images = [Image.open(requests.get(url, stream=True).raw).convert("RGB") for url in image_urls]
images = [Image.open(requests.get(url, stream=True).raw).convert("RGB") for url in image_urls]


question = "whats the difference between these images"

messages = [
    {
        "role": "user",
        "content": [{"type": "image", "image": img} for img in images] + [{"type": "text", "text": question}]
    }
]
print(messages)
# Generate prompt with Qwen-style chat template
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)

# Debug info
print("\n---- DEBUG INFO ----")
print("Generated Prompt:\n")
print(input_text)
print("\nNumber of <|vision_start|> tokens:", input_text.count("<|vision_start|>"))
print("Number of images passed:", len(images))
print("---------------------\n")

# Tokenize inputs
inputs = processor(
    text=[input_text],
    images=images,
    return_tensors="pt"
).to(model.device)

# Clear memory before generation
torch.cuda.empty_cache()
gc.collect()

# Generate response (deterministic)
with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

# Decode only model's output, skipping the prompt
prompt_len = inputs["input_ids"].shape[1]
reply_ids = generate_ids[:, prompt_len:]
output_text = processor.tokenizer.decode(reply_ids[0], skip_special_tokens=True)

# Output
print("\n# Response:\n")
print(output_text)


# Load Datasets

In [9]:
import pandas as pd
cloud=pd.read_excel("../../Dataset/cloud/split_data/cloud_test.xlsx")
device=pd.read_excel("../../Dataset/device/split_data/device_test.xlsx")

## Cloud

In [10]:
cloud.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"['java', 'eclipse', 'google-app-engine', 'java...",https://stackoverflow.com/questions/49711724/s...,4,2018-04-07 20:34:12Z,1,"[""@code511788465541441, if you consider Chanse...",[{'body': '\n \nIf that happens...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,"['azure-web-app-service', 'azure-deployment', ...",https://stackoverflow.com/questions/65255839/f...,7,2020-12-11 17:20:22Z,2,"[""thanks for the hint. i am going to verify th...","[{'body': ""\nI am almost certain that the pack...",\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [11]:
cloud.shape

(1622, 11)

## Processed

In [17]:
cloud=cloud[['title','body','selected_answer','images']]

In [18]:
cloud.head(2)

,title,body,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [19]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()

In [20]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer'],
        'img':cloud['images']
    }
)

In [21]:
cloud_processed.head(2)

,context,gt,img
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Device

In [22]:
device.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,"['linux', 'iptables', 'linux', 'iptables']",https://serverfault.com/questions/993878/what-...,1,2019-12-01 00:23:16Z,1,"['OK, also please add this to your answer so o...",[{'body': '\nIf some NAT operation needs to be...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,"['elasticsearch', 'logstash', 'elastic-stack',...",https://stackoverflow.com/questions/33787795/l...,0,2015-11-18 18:38:40Z,3,"['could you suggest how I could fix?', 'Your d...",[{'body': '\nOK I finally figure how to get th...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [23]:
device.shape

(993, 11)

## Processed

In [24]:
device=device[['title','body','selected_answer','images']]

In [25]:
device.head(2)

,title,body,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [26]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()

In [27]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer'],
        'img':device['images']
    }
)

In [28]:
device_processed.head(2)

,context,gt,img
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


# Prompt Setup

## Cloud

### Zero Shot

In [29]:
zero_shot = """
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:
"""


In [30]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)

In [31]:
print(cloud_processed['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:



### CoT

In [32]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [33]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [34]:
print(cloud_processed['cot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Lets think step by step:
Answer:



## Device

### Zero Shot

In [35]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:

'''

In [36]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)


In [37]:
print(device_processed['zero-shot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Answer:




### CoT

In [38]:
cot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [39]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)

In [40]:
print(device_processed['cot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Lets think step by step:
Answer:



## Prompt Length

In [41]:
# extract prompt length
def prompt_length(text, add_special_tokens=True):
    if pd.isnull(text) or not isinstance(text, str) or not text.strip():
        return 0
    temp_inputs = processor.tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens)
    return temp_inputs.input_ids.shape[1]


In [42]:
# Prompt Length -> Cloud Logs
cloud_processed['zero-shot-question-length'] = cloud_processed['zero-shot-question'].apply(prompt_length)
cloud_processed['cot-question-length'] = cloud_processed['cot-question'].apply(prompt_length)

In [43]:
# Prompt Length -> Device Logs
device_processed['zero-shot-question-length'] = device_processed['zero-shot-question'].apply(prompt_length)
device_processed['cot-question-length'] = device_processed['cot-question'].apply(prompt_length)

# Message Setup ( Prompt + Images )

In [44]:
import ast

def setup_messages(dataset, prompt_column="", image_column="img"):
    messages = []

    for _, row in dataset.iterrows():
        imgs = row.get(image_column, [])
        if isinstance(imgs, str):
            try:
                imgs = ast.literal_eval(imgs)
            except Exception:
                imgs = []

        if imgs:
            # Build structured message with image placeholders and direct question
            content = [{"type": "image", "image": img} for img in imgs if img]
            content.append({
                "type": "text",
                "text": row.get(prompt_column, "")
            })

            messages.append({
                "role": "user",
                "qa": content
            })

    return messages


In [45]:
messages = setup_messages(device_processed,prompt_column="cot-question")

In [46]:
messages[1]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/wM2pW.png'},
  {'type': 'text',
   'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Logstash JDBC input, filter event timestamp different to @timestamp\nbody: I am trying to change the @timestamp to match my timestamp, this is the time my event occurred at not when logstash time stamped it.\n\nhere is my conf file\n\ninput {\n jdbc {\n   jdbc_driver_library => "C:/logstash/lib/mysql-connector-java-5.1.37-bin.jar"\n   jdbc_driver_class => "com.mysql.jdbc.Driver"\n   jdbc_connection_string => "jdbc:mysql://127.0.0.1:3306/transport"\n   jdbc_user => "root"\n   jdbc_password => ""\n   schedule => "* * * * *"\n   stat

# Generate Response

## Setup Model

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

You shouldn't move a model that is dispatched using accelerate hooks.


RuntimeError: You can't move a model that has some modules offloaded to cpu or disk.

In [62]:
import math
from PIL import Image

MAX_LONG_EDGE = 224   
PATCH_SIZE    = 14    
MIN_EDGE      = 28   

def img_resize(img: Image.Image) -> Image.Image:
    w, h = img.size

    # Down-scale if long edge > 224
    scale = min(1.0, MAX_LONG_EDGE / max(w, h))
    w, h = int(round(w * scale)), int(round(h * scale))

    #  Guarantee minimum size
    w, h = max(w, MIN_EDGE), max(h, MIN_EDGE)

    # nearest multiple of patch size
    w  = math.ceil(w / PATCH_SIZE) * PATCH_SIZE
    h  = math.ceil(h / PATCH_SIZE) * PATCH_SIZE

    return img.resize((w, h), Image.Resampling.LANCZOS)


In [63]:
import torch
import gc
import traceback
import requests
import json
import os
from PIL import Image

torch.set_grad_enabled(False)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

failed_responses = []

# Load and prepare image(s) from message
def process_vision_info(messages):
    images = []
    for msg in messages:
        for item in msg.get("content", []):
            if item["type"] == "image":
                img = item["image"]
                if isinstance(img, str):
                    img = Image.open(requests.get(img, stream=True).raw).convert("RGB")
                images.append(img_resize(img))
    return images

# Process message with debug info and collect failed responses
def process_message(msg: dict, idx: int):
    try:
        prompt_str = processor.apply_chat_template(
            [msg], tokenize=False, add_generation_prompt=True
        )

        imgs = process_vision_info([msg])
        if not imgs:
            raise ValueError("No images found in message.")

        inputs = processor(
            text=[prompt_str],
            images=imgs,
            return_tensors="pt"
        ).to(model.device)

        gen_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=False,
            use_cache=True
        )

        reply_ids = gen_ids[:, inputs.input_ids.shape[-1]:]
        answer = processor.batch_decode(
            reply_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return {"idx": idx, "response": answer}

    except Exception as e:
        failed_responses.append({
            "idx": idx,
            "question": msg.get("content", ""),
            "error": str(e),
            "traceback": traceback.format_exc()
        })
        return {"idx": idx, "response": f"[ERROR] {e}"}

    finally:
        for name in ("inputs", "gen_ids"):
            if name in locals():
                del locals()[name]
        torch.cuda.empty_cache()
        gc.collect()

## Testing Samples

In [46]:
#sample=cloud_processed[:5]
sample=cloud_processed[:5]

In [47]:
sample.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,393,399


In [48]:
# Setup messages for model
sample_messages = setup_messages(sample,prompt_column="zero-shot-question")
#sample_messages = setup_messages(sample,prompt_column="cot-question")

In [49]:
sample_messages[:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]}]

### Get response for Sample  -> Test

In [50]:
# Output setup
sample_results = []
success_output_path = "sample_results_full.jsonl"
failure_output_path = "failed_response_sample.jsonl"

sample["generated_response"] = None

# Clear output files first
with open(success_output_path, "w"): pass
with open(failure_output_path, "w"): pass

# Process each message
for row_idx, msg in enumerate(tqdm(sample_messages, desc="Processing Messages")):
    try:
        # Build message
        qa_content = msg.get("qa", [])
        message = {
            "role": msg.get("role", "user"),
            "content": qa_content
        }

        # Extract image URLs
        image_urls = [c.get("image") for c in qa_content if c.get("type") == "image"]

        # Run inference
        result = process_message(message, idx=row_idx)
        response_text = result["response"]

    except Exception as e:
        # Handle unexpected exception
        response_text = f"[ERROR] {e}"
        image_urls = [c.get("image") for c in msg.get("qa", []) if c.get("type") == "image"]
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }

        sample.at[row_idx, "generated_response"] = response_text

        # Save to failed file
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
        continue

    # Update DataFrame
    sample.at[row_idx, "generated_response"] = response_text

    # Print first sample
    if row_idx == 0:
        print("\n#Prompt:##\n")
        print(message["content"])
        print("\n##Sample Response:##\n")
        print(response_text)

    # Build row
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    sample_results.append(row_data)

    # Split output based on success/failure
    if response_text.startswith("[ERROR]"):
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": response_text,
            "traceback": result.get("traceback", ""),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
    else:
        with open(success_output_path, "a") as f:
            f.write(json.dumps(row_data) + "\n")


/tmp/ipykernel_791/1180642016.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["generated_response"] = None
Processing Messages:  20%|██        | 1/5 [00:42<02:51, 42.81s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]

##Sample Response:##

The error message "Some recently changed resources are not yet published" is a common issue in Google App Engine (GAE) development. This error occurs when the GAE server is unable to detect changes made to the project's resources, such as Java classes, HTML fil

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages: 100%|██████████| 5/5 [01:52<00:00, 22.54s/it]


In [51]:
# result sample
sample_results[:1]

[{'row': 0,
  'response': 'The error message "Some recently changed resources are not yet published" is a common issue in Google App Engine (GAE) development. This error occurs when the GAE server is unable to detect changes made to the project\'s resources, such as Java classes, HTML files, or other assets, and therefore uses the old code instead of the updated version.\n\nTo resolve this issue, you can try the following steps:\n\n1. **Clean and rebuild the project**: Go to Project > Clean in Eclipse, then select the project and click Clean. This will delete all temporary files and rebuild the project.\n2. **Restart Eclipse**: Sometimes, a simple restart of Eclipse can resolve the issue.\n3. **Delete the .metadata directory**: The .metadata directory contains configuration files for Eclipse. Deleting it may resolve the issue. However, be aware that this will delete all your project settings, so make sure to back up your project before doing so.\n4. **Check the project\'s configuration

In [52]:
sample['zero-shot-prompt'] = setup_messages(sample, prompt_column="zero-shot-question")

/tmp/ipykernel_791/536900671.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample['zero-shot-prompt'] = setup_messages(sample, prompt_column="zero-shot-question")


In [53]:
sample.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,generated_response,zero-shot-prompt
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127,"The error message ""Some recently changed resou...","{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,393,399,[ERROR] CUDA out of memory. Tried to allocate ...,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [54]:
sample.to_csv("sample_vlm_llama_3_2_vl_11b.csv", index=False)

## Cloud

In [48]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,390,396


### Zero-shot

In [58]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="zero-shot-question")

In [59]:
cloud_messages[:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]}]

In [60]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero_shot_generated_response
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127,None
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,390,396,None


In [67]:
cloud_messages[1515]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/n8Vw7.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/EdDPP.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/pliqf.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/ITnjX.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/Ms3Vz.jpg'},
  {'type': 'image', 'image': 'https://i.sstatic.net/bw6Ui.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/KI2Dn.png'},
  {'type': 'text',
   'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Visual Studio Team Services Continuous Integration: NuGet Restore Task Failed\nbody: I am using Continuous Integration feature in Team Services (was Visual Studio Online). My build definition targets a specific project in a solution (not the whole solution), which is ClientU

#### Get response

In [68]:
# Output setup
cloud_logs_zero_shot_results = []
success_output_path ="vlm_cloud_llama_3_2_vl_11b_zero_shot_results.jsonl"
failure_output_path = "failed_response_cloud_zero_shot.jsonl"

cloud_processed["zero_shot_generated_response"] = None

# Clear output files first
with open(success_output_path, "w"): pass
with open(failure_output_path, "w"): pass

start_index= 1514
# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages[1514:], desc="Processing Messages")):
    try:
        # Build message
        qa_content = msg.get("qa", [])
        message = {
            "role": msg.get("role", "user"),
            "content": qa_content
        }

        # Extract image URLs
        image_urls = [c.get("image") for c in qa_content if c.get("type") == "image"]

        # Run inference
        result = process_message(message, idx=row_idx)
        response_text = result["response"]

    except Exception as e:
        # Handle unexpected exception
        response_text = f"[ERROR] {e}"
        image_urls = [c.get("image") for c in msg.get("qa", []) if c.get("type") == "image"]
        failed_row = {
            "row": row_idx + start_index,
            "response": response_text,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }

        cloud_processed.at[row_idx, "zero_shot_generated_response"] = response_text

        # Save to failed file
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
        continue

    # Update DataFrame
    cloud_processed.at[row_idx, "zero_shot_generated_response"] = response_text

    # Print first sample
    if row_idx == 0:
        print("\n#Prompt:##\n")
        print(message["content"])
        print("\n##Sample Response:##\n")
        print(response_text)

    # Build row
    row_data = {
        "row": row_idx + start_index,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_zero_shot_results.append(row_data)

    # Split output based on success/failure
    if response_text.startswith("[ERROR]"):
        failed_row = {
            "row": row_idx + start_index,
            "response": response_text,
            "error": response_text,
            "traceback": result.get("traceback", ""),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
    else:
        with open(success_output_path, "a") as f:
            f.write(json.dumps(row_data) + "\n")


Processing Messages:   1%|          | 1/108 [01:48<3:13:59, 108.78s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/dc8dP.png'}, {'type': 'image', 'image': 'https://i.sstatic.net/9QXJW.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Run "node test" as part of Visual Studio Team Services build task with results in "tests" tab\nbody: I have a project that contains tests that I am running with Mocha from the command line. I have set up a test script in my packages.json, which looks as follows:\n\n"test": "mocha ./**/*.spec.js --reporter dot --require jsdom-global/register"\n\nI have currently got a simple task set up in Visual Studio Team Services, which just runs the npm test command, this runs Mocha within a console and continues/fails the build depending on whether the tests pass.\n\nWhat I\'d like to be able to do is have the results of m

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages: 100%|██████████| 108/108 [2:14:11<00:00, 74.55s/it]


In [69]:
cloud_processed['zero-shot-prompt'] = setup_messages(cloud_processed, prompt_column="zero-shot-question")

In [70]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero_shot_generated_response,zero-shot-prompt
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127,"Yes, it is possible to publish test results in...","{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,390,396,[ERROR] CUDA out of memory. Tried to allocate ...,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [60]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_llama_3_1_8b_instruct.xlsx", index=False)

### CoT

In [72]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="cot-question")

In [73]:
cloud_messages[0:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nLets think step by step:\nAnswer:\n'}]}]

In [74]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero_shot_generated_response,zero-shot-prompt
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,121,127,"Yes, it is possible to publish test results in...","{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,390,396,[ERROR] CUDA out of memory. Tried to allocate ...,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [77]:
cloud_messages[1000]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/k6Xia.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/NocKP.png'},
  {'type': 'text',
   'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: AWS AppSync: pass arguments from parent resolver to children\nbody: In AWS AppSync, arguments send on the main query don\'t seem to be forwarded to all children resolvers.\ntype Query {\n  article(id: String!, consistentRead: Boolean): Article\n  book(id: String!, consistentRead: Boolean): Book\n}\n\ntype Article {\n  title: String!\n  id: String!\n}\n\ntype Book {\n  articleIds: [String]!\n  articles: [Article]!\n  id: String!\n}\n\nwhen I call:\nquery GetBook {\n  book(id: 123, consistentRead: true) {\n    articles {\n      title\n    }\n  }\n}\n\nthe first query to get the book receives the c

#### Get response

In [ ]:
# Output setup
cloud_logs_cot_results = []
success_output_path ="vlm_cloud_llama_3_2_vl_11b_cot_results.jsonl"
failure_output_path = "failed_response_cloud_cot.jsonl"

cloud_processed["cot_generated_response"] = None

# Clear output files first
with open(success_output_path, "w"): pass
with open(failure_output_path, "w"): pass
start_index= 1000
# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages[1000:], desc="Processing Messages")):
    try:
        # Build message
        qa_content = msg.get("qa", [])
        message = {
            "role": msg.get("role", "user"),
            "content": qa_content
        }

        # Extract image URLs
        image_urls = [c.get("image") for c in qa_content if c.get("type") == "image"]

        # Run inference
        result = process_message(message, idx=row_idx)
        response_text = result["response"]

    except Exception as e:
        # Handle unexpected exception
        response_text = f"[ERROR] {e}"
        image_urls = [c.get("image") for c in msg.get("qa", []) if c.get("type") == "image"]
        failed_row = {
            "row": row_idx + start_index,
            "response": response_text,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }

        cloud_processed.at[row_idx, "cot_generated_response"] = response_text

        # Save to failed file
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
        continue

    # Update DataFrame
    cloud_processed.at[row_idx, "cot_generated_response"] = response_text

    # Print first sample
    if row_idx == 0:
        print("\n#Prompt:##\n")
        print(message["content"])
        print("\n##Sample Response:##\n")
        print(response_text)

    # Build row
    row_data = {
        "row": row_idx + start_index,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_cot_results.append(row_data)

    # Split output based on success/failure
    if response_text.startswith("[ERROR]"):
        failed_row = {
            "row": row_idx + start_index,
            "response": response_text,
            "error": response_text,
            "traceback": result.get("traceback", ""),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
    else:
        with open(success_output_path, "a") as f:
            f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/622 [03:02<31:30:20, 182.64s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/k6Xia.png'}, {'type': 'image', 'image': 'https://i.sstatic.net/NocKP.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: AWS AppSync: pass arguments from parent resolver to children\nbody: In AWS AppSync, arguments send on the main query don\'t seem to be forwarded to all children resolvers.\ntype Query {\n  article(id: String!, consistentRead: Boolean): Article\n  book(id: String!, consistentRead: Boolean): Book\n}\n\ntype Article {\n  title: String!\n  id: String!\n}\n\ntype Book {\n  articleIds: [String]!\n  articles: [Article]!\n  id: String!\n}\n\nwhen I call:\nquery GetBook {\n  book(id: 123, consistentRead: true) {\n    articles {\n      title\n    }\n  }\n}\n\nthe first query to get the book receives the consistentRead para

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:  15%|█▍        | 92/622 [2:55:31<21:36:03, 146.72s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
cloud_processed['cot-prompt'] = setup_messages(cloud_processed, prompt_column="cot-question")

In [ ]:
cloud_processed.head(2)

In [68]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_llama_3_1_8b_instruct_cot.xlsx", index=False)

## Device

In [45]:
device_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,151,157
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,557,563


### Zero-Shot

In [46]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="zero-shot-question")

In [47]:
device_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
   {'ty

#### Get response

In [ ]:
# Output setup
device_logs_zero_shot_results = []
success_output_path ="vlm_device_llama_3_2_vl_11b_zero_shot_results.jsonl"
failure_output_path = "failed_response_device_zero_shot.jsonl"

device_processed["zero_shot_generated_response"] = None

# Clear output files first
with open(success_output_path, "w"): pass
with open(failure_output_path, "w"): pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages, desc="Processing Messages")):
    try:
        # Build message
        qa_content = msg.get("qa", [])
        message = {
            "role": msg.get("role", "user"),
            "content": qa_content
        }

        # Extract image URLs
        image_urls = [c.get("image") for c in qa_content if c.get("type") == "image"]

        # Run inference
        result = process_message(message, idx=row_idx)
        response_text = result["response"]

    except Exception as e:
        # Handle unexpected exception
        response_text = f"[ERROR] {e}"
        image_urls = [c.get("image") for c in msg.get("qa", []) if c.get("type") == "image"]
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }

        device_processed.at[row_idx, "zero_shot_generated_response"] = response_text

        # Save to failed file
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
        continue

    # Update DataFrame
    device_processed.at[row_idx, "zero_shot_generated_response"] = response_text

    # Print first sample
    if row_idx == 0:
        print("\n#Prompt:##\n")
        print(message["content"])
        print("\n##Sample Response:##\n")
        print(response_text)

    # Build row
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_zero_shot_results.append(row_data)

    # Split output based on success/failure
    if response_text.startswith("[ERROR]"):
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": response_text,
            "traceback": result.get("traceback", ""),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
    else:
        with open(success_output_path, "a") as f:
            f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/993 [00:15<4:12:52, 15.29s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'}, {'type': 'text', 'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]

##Sample Response:##

The nat:INPUT chain in the nat table is used for the translation of the source addresses (SNAT) of outgoing packets. This is typically used when a host behind a N

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:  16%|█▋        | 162/993 [1:11:03<5:21:10, 23.19s/it]/usr/local/lib/python3.10/dist-packages/PIL/Image.py:979: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:  56%|█████▌    | 556/993 [4:07:51<4:43:31, 38.93s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
device_logs_zero_shot_results[:2]

In [ ]:
device_processed['zero-shot-prompt'] = setup_messages(device_processed, prompt_column="zero-shot-question")


In [ ]:
device_processed.head(2)

In [76]:
device_processed.to_excel("../responses/device/vlm_response_device_llama_3_1_8b_instruct.xlsx", index=False)

### CoT

In [77]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="cot-question")

In [78]:
device_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\r\n\r\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\r\n\r\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\r\n\r\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nLets think step by step:\nAnswer:\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://

#### Get response

In [ ]:
# Output setup
device_logs_cot_results = []
success_output_path ="vlm_device_llama_3_2_vl_11b_cot_results.jsonl"
failure_output_path = "failed_response_device_cot.jsonl"

device_processed["cot_generated_response"] = None

# Clear output files first
with open(success_output_path, "w"): pass
with open(failure_output_path, "w"): pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages[, desc="Processing Messages")):
    try:
        # Build message
        qa_content = msg.get("qa", [])
        message = {
            "role": msg.get("role", "user"),
            "content": qa_content
        }

        # Extract image URLs
        image_urls = [c.get("image") for c in qa_content if c.get("type") == "image"]

        # Run inference
        result = process_message(message, idx=row_idx)
        response_text = result["response"]

    except Exception as e:
        # Handle unexpected exception
        response_text = f"[ERROR] {e}"
        image_urls = [c.get("image") for c in msg.get("qa", []) if c.get("type") == "image"]
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }

        device_processed.at[row_idx, "cot_generated_response"] = response_text

        # Save to failed file
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
        continue

    # Update DataFrame
    device_processed.at[row_idx, "cot_generated_response"] = response_text

    # Print first sample
    if row_idx == 0:
        print("\n#Prompt:##\n")
        print(message["content"])
        print("\n##Sample Response:##\n")
        print(response_text)

    # Build row
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_cot_results.append(row_data)

    # Split output based on success/failure
    if response_text.startswith("[ERROR]"):
        failed_row = {
            "row": row_idx,
            "response": response_text,
            "error": response_text,
            "traceback": result.get("traceback", ""),
            "question": msg.get("qa", []),
            "image_urls": image_urls
        }
        with open(failure_output_path, "a") as f:
            f.write(json.dumps(failed_row) + "\n")
    else:
        with open(success_output_path, "a") as f:
            f.write(json.dumps(row_data) + "\n")


In [ ]:
device_logs_cot_results[:2]

[{'row': 0,
  'response': 'The INPUT chain in the nat table is typically used for handling packets destined for the local host itself. This includes packets that are not destined for any specific port but are instead handled by the kernel\'s routing code or other system components. The PREROUTING chain, on the other hand, is specifically designed for translating the destination address of incoming packets, which is commonly referred to as "port forwarding." Therefore, the INPUT chain in the nat table serves a different purpose than the PREROUTING chain.',
  'image_urls': ['https://i.sstatic.net/NUh2k.png']},
 {'row': 1,
  'response': 'The issue arises because the `@timestamp` field in Elasticsearch is being parsed incorrectly due to the format mismatch between your `timestamp` field and the expected format for parsing. The `@timestamp` field is being set to a default value (`2015-11-18T18:22:00.069Z`) which is not matching the actual `timestamp` value in the document.\n\nHere’s how you

In [ ]:
device_processed['cot-prompt'] = setup_messages(device_processed, prompt_column="cot-question")


In [ ]:
device_processed.head(2)

In [ ]:
device_processed.to_excel("../responses/device/vlm_response_device_llama_3_1_8b_instruct_cot.xlsx", index=False)